# EMS Value Analytics - Medicaid Insights

Prepared for the Noah Smith discussion. The question behind all of it: **did GMR get the patient to the right level of care, at the right time, and what does the Medicaid population look like where we serve it.**

The Mississippi preliminary slide put Medicaid at roughly 30% of patients, 88% lower acuity, and 75% scheduled transfers, with several disease states running well above GMR's national rate. This notebook reproduces those numbers from the ePCR data, puts a national baseline underneath them so "above national" has a measured denominator, and extends the same cuts to every state, county, and agency.

Data: ImageTrend Elite, GMR ground (`prod.silver_frn_qry_elite_dwgmr_repl`), incidents 2024 to 2026. Counts are records, meaning calls, not distinct people.

**Compute: use the SQL Pro warehouse.** The old catalog was removed and the federated one behaves differently on other compute.

## 1. Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6"

OUT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/ems_value/results"
os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

FOCUS_STATE = "Mississippi"
MIN_STATE_INCIDENTS = 1000

RESULTS = {}
def keep(d, n):
    RESULTS[n] = d.copy()
    return d

print("run:", RUN_ID)

`FOCUS_STATE` drives the Mississippi cuts. Change it to run the same slide for another state without touching anything else.

## 2. Confirm the tables and columns

In [ ]:
for t in ["fact_incident", "dim_incident", "dim_situation", "dim_disposition",
          "dim_agency", "dim_patient", "dim_payment", "dim_scene"]:
    try:
        cols = [f.name for f in spark.table(f"prod.silver_frn_qry_elite_dwgmr_repl.{t}").schema.fields]
        print(t, len(cols), "columns")
    except Exception as e:
        print(t, str(e)[:140])

In [ ]:
for t, keys in [("dim_patient", ["age", "gender", "sex", "race", "longitudinal", "record_number"]),
                ("dim_incident", ["service", "type", "acuity", "priority", "complaint"]),
                ("dim_disposition", ["destination", "transport", "level_of_care", "disposition"])]:
    try:
        cols = [f.name for f in spark.table(f"prod.silver_frn_qry_elite_dwgmr_repl.{t}").schema.fields]
        print(t)
        print([c for c in cols if any(k in c.lower() for k in keys)])
        print()
    except Exception as e:
        print(t, str(e)[:140], "\n")

The SQL below names columns explicitly. Four of them are best guesses carried over from the old catalog and should be confirmed here before trusting the query that uses them:

- `Patient_Age` and `Patient_Gender` on `dim_patient` - used by the demographics cut
- `Incident_Type_Of_Service_Requested` on `dim_incident` - used by the scheduled transfer split
- `Incident_Dispatch_Priority_Patient_Acuity` on `dim_incident` - used by the acuity cut

Each query is independent, so a wrong name costs one result rather than the run. The second cell also looks for a longitudinal record number, which is what would let us measure repeat patients properly.

## 3. Run the queries

In [ ]:
QUERIES = {

"control_totals": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT yr,
       count(*) AS incidents,
       count(DISTINCT incident_id) AS distinct_incidents,
       sum(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END) AS medicaid,
       round(100.0 * avg(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END), 1) AS medicaid_pct
FROM scope GROUP BY yr ORDER BY yr
""",

"payer_by_year": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT yr, payer_group, count(*) AS incidents
FROM scope GROUP BY yr, payer_group ORDER BY yr, incidents DESC
""",

"payer_values": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT payer, count(*) AS incidents,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM scope GROUP BY payer ORDER BY incidents DESC LIMIT 40
""",

"medicaid_by_state": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state,
       count(*) AS incidents,
       sum(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END) AS medicaid,
       round(100.0 * avg(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END), 1) AS medicaid_pct
FROM scope GROUP BY state HAVING count(*) >= 1000 ORDER BY incidents DESC
""",

"medicaid_by_county": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, county,
       count(*) AS incidents,
       sum(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END) AS medicaid,
       round(100.0 * avg(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END), 1) AS medicaid_pct
FROM scope GROUP BY state, county HAVING count(*) >= 500 ORDER BY medicaid DESC LIMIT 300
""",

"medicaid_by_agency": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, agency_name,
       count(*) AS incidents,
       sum(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END) AS medicaid,
       round(100.0 * avg(CASE WHEN payer_group = 'Medicaid' THEN 1 ELSE 0 END), 1) AS medicaid_pct
FROM scope GROUP BY state, agency_name HAVING count(*) >= 300 ORDER BY medicaid_pct DESC LIMIT 200
""",

"acuity_by_payer": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, payer_group, acuity_band, count(*) AS incidents
FROM scope GROUP BY state, payer_group, acuity_band
""",

"call_origin_by_payer": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, payer_group, call_origin, count(*) AS incidents
FROM scope GROUP BY state, payer_group, call_origin
""",

"service_requested_values": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT service_requested, count(*) AS incidents
FROM scope GROUP BY service_requested ORDER BY incidents DESC LIMIT 40
""",

"disposition_by_payer": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT payer_group, acuity_band, disposition_group, count(*) AS incidents
FROM scope GROUP BY payer_group, acuity_band, disposition_group
""",

"impression_national": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT primary_impression, count(*) AS medicaid_incidents
FROM med GROUP BY primary_impression ORDER BY medicaid_incidents DESC LIMIT 400
""",

"impression_by_state": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, primary_impression, count(*) AS medicaid_incidents
FROM med GROUP BY state, primary_impression HAVING count(*) >= 20
""",

"demographics": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT state, payer_group, patient_gender,
       CASE
         WHEN patient_age < 10 THEN '0-9'
         WHEN patient_age < 20 THEN '10-19'
         WHEN patient_age < 40 THEN '20-39'
         WHEN patient_age < 60 THEN '40-59'
         WHEN patient_age < 80 THEN '60-79'
         WHEN patient_age >= 80 THEN '80+'
         ELSE 'Not recorded'
       END AS age_band,
       count(*) AS incidents
FROM scope GROUP BY state, payer_group, patient_gender, 4
""",

"hour_by_day": """
WITH flat AS (
  SELECT
    fi.Incident_Transaction_GUID_Internal          AS incident_id,
    inc.Incident_Date_Time                         AS incident_date,
    pay.Payment_Primary_Method_Of_Payment          AS payer,
    inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
    inc.Incident_Type_Of_Service_Requested         AS service_requested,
    sit.Situation_Provider_Primary_Impression      AS primary_impression,
    dis.Disposition_Incident_Patient_Disposition   AS disposition,
    pat.Patient_Age                                AS patient_age,
    pat.Patient_Gender                             AS patient_gender,
    sce.Scene_Incident_State_Name                  AS state,
    fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
    ag.Agency_Name                                 AS agency_name
  FROM prod.silver_frn_qry_elite_dwgmr_repl.fact_incident fi
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
  LEFT JOIN prod.silver_frn_qry_elite_dwgmr_repl.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
),
scope AS (
  SELECT *,
    year(incident_date) AS yr,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur' THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ('', 'null', 'not recorded', 'not applicable', 'not reporting', 'unknown') THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    CASE
      WHEN lower(acuity) RLIKE 'critical|red|emergent' THEN 'High'
      WHEN lower(acuity) RLIKE 'urgent|yellow'         THEN 'Medium'
      WHEN lower(acuity) RLIKE 'lower|low|green|non-acute|minor' THEN 'Low'
      ELSE 'Not recorded'
    END AS acuity_band,
    CASE
      WHEN lower(service_requested) RLIKE 'interfacility|inter-facility|transfer|scheduled' THEN 'Scheduled transfer'
      WHEN lower(service_requested) RLIKE '911|emergency|scene'                             THEN '911 response'
      ELSE 'Other/Not recorded'
    END AS call_origin,
    CASE
      WHEN lower(disposition) RLIKE 'transport' THEN 'Transported'
      WHEN lower(disposition) RLIKE 'refus'     THEN 'Refused'
      WHEN lower(disposition) RLIKE 'no patient|cancel|no treatment' THEN 'No patient / cancelled'
      WHEN disposition IS NULL OR lower(trim(disposition)) IN ('', 'not recorded', 'not applicable') THEN 'Not recorded'
      ELSE 'Other'
    END AS disposition_group
  FROM flat
  WHERE year(incident_date) BETWEEN 2024 AND 2026
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
SELECT date_format(incident_date, 'E') AS dow,
       hour(incident_date) AS hr,
       count(*) AS incidents
FROM med GROUP BY 1, 2
""",

}

results = {}
failed = {}
for name, sql in QUERIES.items():
    try:
        results[name] = spark.sql(sql).toPandas()
        print(f"{name}: {len(results[name]):,} rows")
    except Exception as e:
        failed[name] = str(e)
        print(f"{name}: FAILED - {str(e)[:160]}")

Every query is a complete standalone statement carrying its own `WITH flat ... scope ... med` block, so any one of them can be copied into the SQL editor and run on its own.

**The joins.** `fact_incident` holds foreign keys; the readable values come from the dimensions, each joined `Dim_X_FK` to `Dim_X_PK`. All LEFT, so an incident with no payment row still appears with a null payer.

**The groupings**, all applied in `scope` so every cut uses the same definitions:

| Group | Rule |
|---|---|
| payer_group | Medicaid, Medicare, Self Pay, Unknown, Other/Commercial - first match wins |
| acuity_band | High (critical, red, emergent), Medium (urgent, yellow), Low (lower, green, non-acute, minor) |
| call_origin | Scheduled transfer, 911 response, Other |
| disposition_group | Transported, Refused, No patient / cancelled, Other |

Medicaid here matches only explicit Medicaid, CHIP, and Title XIX values. Managed care plan names are deliberately not swept in, which makes every Medicaid figure below a floor rather than an estimate. Section 4 shows how much room sits in the unclassified values.

## 4. What the payer field can and cannot tell us

In [ ]:
pv = keep(results["payer_values"], "payer_values")
display(pv.head(20))

ct = keep(results["control_totals"], "control_totals")
display(ct)

Read the raw payer values before any Medicaid number leaves this notebook.

If a generic bucket such as "Insurance" or a large blank share sits near the top, the real payer for those records is unknown and the Medicaid share is understated by an amount we cannot measure. That is the single most important caveat to carry into the meeting: our Medicaid counts are a floor.

The control totals also separate records from people. These are calls, not distinct patients, and a patient with three transports counts three times.

In [ ]:
t = keep(results["payer_by_year"], "payer_by_year")
p = t.pivot(index="yr", columns="payer_group", values="incidents").fillna(0)
p = p.div(p.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10,4.5))
bottom = np.zeros(len(p))
for i, c in enumerate(p.columns):
    ax.bar(p.index.astype(str), p[c], bottom=bottom, label=c,
           color=[NAVY, TEAL, GOLD, CORAL, GREY][i % 5])
    bottom += p[c].values
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Payer mix by year, share of ePCR records")
ax.legend(fontsize=9, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.08))
plt.tight_layout(); plt.show()
display(p.round(1))

Payer mix over time. A rising Unknown share in the most recent year usually means payer is written at billing rather than at the scene, so recent months look unclassified regardless of who actually paid.

## 5. Where the Medicaid volume is

In [ ]:
st = keep(results["medicaid_by_state"], "medicaid_by_state")
top = st.sort_values("medicaid_pct", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10,7))
o = top.sort_values("medicaid_pct")
colors = [CORAL if s == FOCUS_STATE else TEAL for s in o["state"]]
ax.barh(range(len(o)), o["medicaid_pct"], color=colors)
ax.set_yticks(range(len(o))); ax.set_yticklabels(o["state"], fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Medicaid share of ePCR records by state (states with 1,000+ records)")
for i, v in enumerate(o["medicaid_pct"]):
    ax.annotate(f"{v:.1f}%", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
plt.tight_layout(); plt.show()
display(st.head(25))

Medicaid share by state, highest first, with the focus state highlighted.

This is the frame the Mississippi slide was missing. A 30% Medicaid rate means something different depending on whether the national figure is 5% or 25%, and depending on whether Mississippi sits at the top of this list or in the middle.

In [ ]:
cty = keep(results["medicaid_by_county"], "medicaid_by_county")
display(cty.head(25))

ag = keep(results["medicaid_by_agency"], "medicaid_by_agency")
display(ag.head(25))

County and agency views of the same measure.

The Mississippi slide named MS-MEDSTAT-Ground at 52.1% and MS-PANOLA LGA-Ground at 61.4%. The agency table puts those rates next to every other agency, which turns "these two are high" into "these two rank here nationally". Agencies with a high Medicaid rate and high volume are where a payer conversation or a navigation pilot would start.

County is also the join level for ACS, HPSA and CDC PLACES, so this table is what any community or access overlay would attach to.

## 6. Acuity - is this really a low acuity population

In [ ]:
ac = keep(results["acuity_by_payer"], "acuity_by_payer")

nat = ac.groupby(["payer_group","acuity_band"])["incidents"].sum().unstack(fill_value=0)
nat_pct = (nat.div(nat.sum(axis=1), axis=0) * 100).round(1)

foc = ac[ac["state"] == FOCUS_STATE].groupby(["payer_group","acuity_band"])["incidents"].sum().unstack(fill_value=0)
foc_pct = (foc.div(foc.sum(axis=1), axis=0) * 100).round(1)

print("National"); display(nat_pct)
print(FOCUS_STATE); display(foc_pct)
keep(nat_pct.reset_index(), "acuity_national")
keep(foc_pct.reset_index(), "acuity_focus_state")

In [ ]:
bands = [b for b in ["Low","Medium","High","Not recorded"] if b in nat_pct.columns]
groups = [g for g in ["Medicaid","Medicare","Self Pay","Other/Commercial"] if g in nat_pct.index]
x = np.arange(len(groups)); w = 0.8 / len(bands)

fig, ax = plt.subplots(figsize=(10,4.5))
for i, b in enumerate(bands):
    ax.bar(x + i*w, nat_pct.loc[groups, b], w, label=b, color=[TEAL, GOLD, CORAL, GREY][i % 4])
ax.set_xticks(x + w*(len(bands)-1)/2); ax.set_xticklabels(groups)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Acuity band by payer, national")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

The 88% lower acuity figure is the one most likely to be quoted back at us, so it needs the comparison beside it.

Two questions it has to survive. Is Medicaid actually lower acuity than the other payer groups, or is all of EMS mostly lower acuity? And how much of the acuity field is "Not recorded", since a large unrecorded share makes any acuity percentage fragile.

The national literature argues that treating all Medicaid encounters as low acuity is both clinically unsafe and analytically misleading. If our own data shows the Medicaid acuity distribution sitting close to the other payer groups, that is a stronger statement than the national argument because it is ours.

## 7. Scheduled transfers versus 911 responses

In [ ]:
sr = keep(results["service_requested_values"], "service_requested_values")
display(sr.head(20))

co = keep(results["call_origin_by_payer"], "call_origin_by_payer")
nat_co = co.groupby(["payer_group","call_origin"])["incidents"].sum().unstack(fill_value=0)
nat_co_pct = (nat_co.div(nat_co.sum(axis=1), axis=0) * 100).round(1)
foc_co = co[co["state"] == FOCUS_STATE].groupby(["payer_group","call_origin"])["incidents"].sum().unstack(fill_value=0)
foc_co_pct = (foc_co.div(foc_co.sum(axis=1), axis=0) * 100).round(1)

print("National"); display(nat_co_pct)
print(FOCUS_STATE); display(foc_co_pct)
keep(nat_co_pct.reset_index(), "call_origin_national")
keep(foc_co_pct.reset_index(), "call_origin_focus_state")

The 75% scheduled transfer figure changes what the Medicaid number means, so it deserves its own check.

If three quarters of Medicaid transports in a market are scheduled interfacility moves, then that market's Medicaid volume is largely a contracted transport business, not a 911 population that nurse navigation or a community paramedicine program could divert. The two need to be reported separately or the opportunity gets sized against the wrong denominator.

Check the raw service requested values first. If the split lands almost entirely in "Other/Not recorded", the classification rule needs the real category names rather than the keyword match used here.

## 8. Conditions that run above the national rate

In [ ]:
nat_imp = keep(results["impression_national"], "impression_national")
st_imp = results["impression_by_state"]

nat_total = nat_imp["medicaid_incidents"].sum()
nat_rate = nat_imp.set_index("primary_impression")["medicaid_incidents"] / nat_total

foc = st_imp[st_imp["state"] == FOCUS_STATE].copy()
foc_total = foc["medicaid_incidents"].sum()
foc["focus_rate"] = foc["medicaid_incidents"] / foc_total
foc["national_rate"] = foc["primary_impression"].map(nat_rate)
foc = foc.dropna(subset=["national_rate"])
foc["index_vs_national"] = (foc["focus_rate"] / foc["national_rate"]).round(2)
foc["focus_pct"] = (foc["focus_rate"] * 100).round(2)
foc["national_pct"] = (foc["national_rate"] * 100).round(2)

over = foc[foc["medicaid_incidents"] >= 50].sort_values("index_vs_national", ascending=False)
over = over[["primary_impression","medicaid_incidents","focus_pct","national_pct","index_vs_national"]]
keep(over, "conditions_over_index")
display(over.head(20))

In [ ]:
o = over.head(15).sort_values("index_vs_national")
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(range(len(o)), o["index_vs_national"], color=CORAL)
ax.axvline(1.0, color=NAVY, lw=1.5)
ax.set_yticks(range(len(o))); ax.set_yticklabels([str(v)[:45] for v in o["primary_impression"]], fontsize=9)
ax.set_title(f"{FOCUS_STATE} Medicaid conditions vs GMR national Medicaid rate (1.0 = national)")
for i, v in enumerate(o["index_vs_national"]):
    ax.annotate(f"{v:.1f}x", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
plt.tight_layout(); plt.show()

This is the "7.4x the national rate" style of finding, computed properly.

For each condition, the share of the focus state's Medicaid records is divided by the share of GMR's national Medicaid records. An index of 1.0 means the state looks like the rest of GMR; 7.4 means the condition appears seven times more often here than it does nationally.

Two guards are built in. Conditions with fewer than 50 records in the state are excluded, because a handful of records can produce an enormous ratio off a small base. And the comparison is Medicaid against Medicaid, so the result is not just restating that Medicaid patients differ from commercial patients.

Read the absolute percentages alongside the index. A condition at 1.56% against 0.21% is a real 7.4x, but it is still under two percent of volume, which matters when sizing a program around it.

## 9. Disposition - the avoidable transport question

In [ ]:
dp = keep(results["disposition_by_payer"], "disposition_by_payer")
t = dp.groupby(["payer_group","disposition_group"])["incidents"].sum().unstack(fill_value=0)
t_pct = (t.div(t.sum(axis=1), axis=0) * 100).round(1)
display(t_pct)
keep(t_pct.reset_index(), "disposition_by_payer_pct")

low = dp[(dp["acuity_band"] == "Low") & (dp["disposition_group"] == "Transported")]
low_by_payer = low.groupby("payer_group")["incidents"].sum()
all_by_payer = dp.groupby("payer_group")["incidents"].sum()
cohort = pd.DataFrame({
    "incidents": all_by_payer,
    "low_acuity_transported": low_by_payer,
    "pct_of_payer_group": (low_by_payer / all_by_payer * 100).round(1)}).reset_index()
keep(cohort, "low_acuity_transported")
display(cohort)

Low acuity and transported is the working definition of the avoidable ED transport cohort: a patient the crew themselves rated as lower acuity who still ended up riding to an emergency department.

This is the first of the six builds on the value analytics list, and the one with the clearest dollar figure attached, because each of these encounters has an ED facility cost behind it that a different destination might have avoided.

Two cautions to state whenever this number is used. Low acuity at the scene is not the same as "did not need the ED" - chest pain and syncope are routinely sent home after a workup, and were correctly transported. And without an outcome record we cannot confirm what happened at the hospital, so this sizes a cohort to investigate rather than a savings estimate.

## 10. Who the Medicaid patients are

In [ ]:
dg = keep(results["demographics"], "demographics")

age_order = ["0-9","10-19","20-39","40-59","60-79","80+","Not recorded"]
nat_age = dg[dg["payer_group"] == "Medicaid"].groupby("age_band")["incidents"].sum().reindex(age_order).fillna(0)
foc_age = dg[(dg["payer_group"] == "Medicaid") & (dg["state"] == FOCUS_STATE)].groupby("age_band")["incidents"].sum().reindex(age_order).fillna(0)

comp = pd.DataFrame({
    "national_pct": (nat_age / nat_age.sum() * 100).round(1),
    f"{FOCUS_STATE.lower()}_pct": (foc_age / foc_age.sum() * 100).round(1)})
keep(comp.reset_index(), "medicaid_age_mix")

x = np.arange(len(age_order))
fig, ax = plt.subplots(figsize=(10,4.5))
ax.bar(x - 0.2, comp["national_pct"], 0.4, label="National Medicaid", color=GREY)
ax.bar(x + 0.2, comp[f"{FOCUS_STATE.lower()}_pct"], 0.4, label=f"{FOCUS_STATE} Medicaid", color=TEAL)
ax.set_xticks(x); ax.set_xticklabels(age_order, fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Medicaid age mix, national versus focus state")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
display(comp)

The Mississippi slide called out an older, female-skewed population with a meaningful 10 to 19 group. This puts that against the national Medicaid mix so the skew can be stated as a difference rather than an observation.

The pediatric and adolescent share matters for a different reason than the older group: those are the encounters most likely to have a primary care or urgent care alternative, and they are the population Medicaid covers most broadly.

## 11. Headline numbers for the discussion

In [ ]:
nat_med_pct = float(results["control_totals"]["medicaid"].sum() / results["control_totals"]["incidents"].sum() * 100)
st_row = results["medicaid_by_state"].set_index("state")
foc_med_pct = float(st_row.loc[FOCUS_STATE, "medicaid_pct"]) if FOCUS_STATE in st_row.index else np.nan
foc_rank = int((st_row["medicaid_pct"] > foc_med_pct).sum() + 1) if FOCUS_STATE in st_row.index else -1

low_nat = float(nat_pct.loc["Medicaid", "Low"]) if "Low" in nat_pct.columns else np.nan
low_foc = float(foc_pct.loc["Medicaid", "Low"]) if "Low" in foc_pct.columns else np.nan
sched_foc = float(foc_co_pct.loc["Medicaid", "Scheduled transfer"]) if "Scheduled transfer" in foc_co_pct.columns else np.nan
unknown_pct = float(results["payer_by_year"].query("payer_group == 'Unknown'")["incidents"].sum() /
                    results["control_totals"]["incidents"].sum() * 100)

headline = pd.DataFrame([
    ("Records analyzed, 2024-2026", f"{results['control_totals']['incidents'].sum():,.0f}"),
    ("Medicaid share, national", f"{nat_med_pct:.1f}%"),
    (f"Medicaid share, {FOCUS_STATE}", f"{foc_med_pct:.1f}%"),
    (f"{FOCUS_STATE} rank among states by Medicaid share", f"{foc_rank}"),
    ("Medicaid low acuity, national", f"{low_nat:.1f}%"),
    (f"Medicaid low acuity, {FOCUS_STATE}", f"{low_foc:.1f}%"),
    (f"Medicaid scheduled transfers, {FOCUS_STATE}", f"{sched_foc:.1f}%"),
    ("Payer not classified", f"{unknown_pct:.1f}%"),
], columns=["measure", "value"])
keep(headline, "headline")
display(headline)

The numbers to put on a slide, each with its national comparison so none of them stand alone.

The last row is the one to say out loud rather than bury. Every Medicaid figure above is a floor, because that share of records carries no usable payer value. If the unclassified share is large, the honest framing is "at least this many" rather than a point estimate.

What this supports asking Noah for: which cohort to build first. Low acuity transported, the scheduled transfer book, or the over-indexed conditions are three different programs with three different owners, and the data supports sizing any of them but not chasing all three at once.

## 12. Write the workbook

In [ ]:
def sanitize(d):
    o = d.copy()
    o.columns = [str(c) for c in o.columns]
    for c in o.columns:
        if o[c].dtype == object or str(o[c].dtype).startswith("category"):
            o[c] = o[c].apply(lambda v: "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    return o

TABS = [("headline","Headline"),("control_totals","Control Totals"),("payer_values","Payer Values"),
        ("payer_by_year","Payer by Year"),("medicaid_by_state","Medicaid by State"),
        ("medicaid_by_county","Medicaid by County"),("medicaid_by_agency","Medicaid by Agency"),
        ("acuity_national","Acuity National"),("acuity_focus_state","Acuity Focus State"),
        ("service_requested_values","Service Requested"),("call_origin_national","Call Origin National"),
        ("call_origin_focus_state","Call Origin Focus State"),("conditions_over_index","Conditions Over Index"),
        ("disposition_by_payer_pct","Disposition by Payer"),("low_acuity_transported","Low Acuity Transported"),
        ("medicaid_age_mix","Medicaid Age Mix")]

xlsx = os.path.join(OUT_DIR, f"EMS_Value_Medicaid_Insights_{RUN_ID}.xlsx")
try:
    import xlsxwriter; eng = "xlsxwriter"
except ImportError:
    eng = "openpyxl"

with pd.ExcelWriter(xlsx, engine=eng) as w:
    pd.DataFrame({"EMS Value Analytics - Medicaid Insights": [
        f"Run {RUN_ID}",
        "Source: prod.silver_frn_qry_elite_dwgmr_repl, ImageTrend Elite GMR ground, 2024-2026",
        "Counts are ePCR records (calls), not distinct patients.",
        f"Focus state: {FOCUS_STATE}",
        "Medicaid matches explicit Medicaid, CHIP and Title XIX values only. Managed care plan names are not included, so Medicaid figures are a floor.",
        "No patient identifiers are included in this file."]}).to_excel(w, sheet_name="About", index=False)
    for key, tab in TABS:
        if key in RESULTS:
            sanitize(RESULTS[key]).to_excel(w, sheet_name=tab[:31], index=False)

print(xlsx)

One workbook, de-identified, counts and rates only. Safe to send ahead of the meeting without a data access request.

## 13. What this cannot answer yet

- **Outcomes.** ESO returns hospital data on roughly 45% of the records it holds, and ESO itself is only about 14.5% of all ePCR records. Elite carries hospital outcomes on well under 1% of incidents. So "did the patient do well at the alternative destination" is not answerable from this notebook today, and none of the ESO PCR numbers appear in Elite, which means the two sources do not join on that key.
- **Repeat patients.** No stable person identifier has been confirmed in this schema. ESO's longitudinal record number is the candidate; section 2 checks whether anything equivalent exists here. Until then the utilization pyramid and return-interval work cannot be built from Elite.
- **True payer.** The unclassified share caps how precise any Medicaid figure can be. Revenue cycle data is expected in about three weeks and should settle whether payer comes from billing or from the crew.
- **Air and ground CCT.** This notebook is GMR ground only. The AMGH catalog holds air plus ground critical care transports mixed together, with no confirmed way to separate them yet.
- **Nurse navigation overlap.** The link between nurse navigation calls and ePCR records is patients who were transported, since those appear in both. That join is what would let us measure whether a routing decision actually worked.